In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)


In [2]:
print("LOGISTIC REGRESSION - COMPLAINT CLASSIFIER")
print("\nSTEP 1: Loading preprocessed dataset...")

data_path = '../data/preprocess_data/complaints_cleaned.csv'
df = pd.read_csv(data_path)
print(f"✓ Loaded: {data_path}")
print(f"  Shape: {df.shape}")


LOGISTIC REGRESSION - COMPLAINT CLASSIFIER

STEP 1: Loading preprocessed dataset...
✓ Loaded: ../data/preprocess_data/complaints_cleaned.csv
  Shape: (1282355, 3)


In [3]:
print("STEP 2: Preparing features and target...")

target_col = "Product"
primary_feature_col = "Issue"
secondary_feature_candidates = ["Sub-product", "Sub-issue"]

secondary_feature_col = next((c for c in secondary_feature_candidates if c in df.columns), None)
if secondary_feature_col is None:
    raise ValueError(f"Missing secondary feature column. Expected one of: {secondary_feature_candidates}")

required_cols = [primary_feature_col, target_col, secondary_feature_col]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print(f"✓ Feature columns: {[primary_feature_col, secondary_feature_col]}")
print(f"✓ Target column: {target_col}")

# Combine both text features into one model input
df[primary_feature_col] = df[primary_feature_col].fillna("").astype(str)
df[secondary_feature_col] = df[secondary_feature_col].fillna("").astype(str)
df[target_col] = df[target_col].fillna("").astype(str)

X_text = (df[primary_feature_col] + " " + df[secondary_feature_col]).str.replace(r"\s+", " ", regex=True).str.strip()
y = df[target_col]

# Keep rows where at least one feature token exists and target is non-empty
valid_mask = (X_text.str.len() > 0) & (y.str.len() > 0)
X_text = X_text[valid_mask]
y = y[valid_mask]

print(f"✓ Rows used for modeling: {len(X_text):,}")
print(f"✓ Unique target classes: {y.nunique()}")


STEP 2: Preparing features and target...
✓ Feature columns: ['Issue', 'Sub-issue']
✓ Target column: Product
✓ Rows used for modeling: 1,282,355
✓ Unique target classes: 18


In [4]:
print("STEP 3: Train-test split...")

X_train, X_test, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size: {len(X_train):,}")
print(f"Test size: {len(X_test):,}")


STEP 3: Train-test split...
Train size: 1,025,884
Test size: 256,471


In [5]:
print("STEP 4: Training Logistic Regression model...")

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.95)),
    ("clf", LogisticRegression(max_iter=1000, solver="lbfgs"))
])

model.fit(X_train, y_train)
print("✓ Model training complete")

# Save the trained pipeline for later reuse
model_dir = "../models/LogisticRegression"
os.makedirs(model_dir, exist_ok=True)
model_path = f"{model_dir}/logistic_regression_model.joblib"
joblib.dump(model, model_path)
print(f"✓ Saved model to: {model_path}")


STEP 4: Training Logistic Regression model...
✓ Model training complete
✓ Saved model to: ../models/LogisticRegression/logistic_regression_model.joblib


In [6]:
print("STEP 5: Evaluating model...")

y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_pred)
train_balanced_accuracy = balanced_accuracy_score(y_train, y_train_pred)
test_balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
train_f1 = f1_score(y_train, y_train_pred, average="weighted")
test_f1 = f1_score(y_test, y_pred, average="weighted")

print(f"\nTrain Accuracy: {train_accuracy * 100:.2f}%")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Train Balanced Accuracy: {train_balanced_accuracy * 100:.2f}%")
print(f"Test Balanced Accuracy: {test_balanced_accuracy * 100:.2f}%")
print(f"Train Weighted F1: {train_f1 * 100:.2f}%")
print(f"Test Weighted F1: {test_f1 * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# Confusion matrix plot
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues")
ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Normalized Confusion Matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
plt.tight_layout()

plot_dir = "../data/plots"
os.makedirs(plot_dir, exist_ok=True)
cm_path = f"{plot_dir}/confusion_matrix.png"
fig.savefig(cm_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved confusion matrix to: {cm_path}")

# Actual vs predicted comparison plot using top classes in test data
comparison = (
    pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
    .melt(var_name="Type", value_name="Product")
    .groupby(["Type", "Product"])
    .size()
    .reset_index(name="Count")
)

top_products = y_test.value_counts().head(10).index
comparison = comparison[comparison["Product"].isin(top_products)]
comparison_pivot = comparison.pivot(index="Product", columns="Type", values="Count").fillna(0)
comparison_pivot = comparison_pivot.reindex(top_products)

comparison_pivot.plot(kind="bar", figsize=(14, 6))
plt.title("Actual vs Predicted Class Counts (Top 10 Test Classes)")
plt.xlabel("Product")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
comparison_path = f"{plot_dir}/actual_vs_predicted.png"
plt.savefig(comparison_path, dpi=200, bbox_inches="tight")
plt.close()
plt.show()
print(f"Saved actual vs predicted plot to: {comparison_path}")


STEP 5: Evaluating model...

Train Accuracy: 98.45%
Test Accuracy: 98.43%
Train Balanced Accuracy: 83.98%
Test Balanced Accuracy: 83.86%
Train Weighted F1: 98.42%
Test Weighted F1: 98.40%

Classification Report:
                                                                              precision    recall  f1-score   support

                                                     Bank account or service       1.00      1.00      1.00     17241
                                                 Checking or savings account       1.00      0.99      1.00      8128
                                                               Consumer Loan       0.95      0.98      0.96      6321
                                                                 Credit card       0.94      1.00      0.97     17838
                                                 Credit card or prepaid card       0.99      0.95      0.97      9530
                                                            Credit reporting   

In [7]:
print("STEP 6: Predicting label for sample feature...")

sample_issue = "incorrect debit card charge"
sample_secondary_feature = "Incorrect charge on my debit card"
sample_feature = f"{sample_issue} {sample_secondary_feature}"

predicted_label = model.predict([sample_feature])[0]

print(f"  {primary_feature_col}: {sample_issue}")
print(f"  {secondary_feature_col}: {sample_secondary_feature}")
print(f"Predicted Product label: {predicted_label}")

print("\nAll final results saved to ../results/Dev")
print(f"Model artifact available at: {model_path}")


STEP 6: Predicting label for sample feature...
  Issue: incorrect debit card charge
  Sub-issue: Incorrect charge on my debit card
Predicted Product label: Credit card or prepaid card

All final results saved to ../results/Dev
Model artifact available at: ../models/LogisticRegression/logistic_regression_model.joblib
